# Lola — Bootstrap en Colab (v2, con todos los fixes)
Corré las celdas EN ORDEN. Este notebook incluye todos los parches
descubiertos durante troubleshooting: pip para python3.8, checkpoint
de un release distinto, parche de basicsr/torchvision, etc.

**Antes de empezar**: Entorno de ejecución > Cambiar tipo de entorno
de ejecución > GPU.

In [ ]:
# Celda 1: confirmar GPU
!nvidia-smi

In [ ]:
# Celda 2: clonar tu repo + SadTalker
!git clone https://github.com/gustavopaine/lola-project.git
%cd lola-project
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker

In [ ]:
# Celda 3: login en Hugging Face
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# Celda 4: instalar pip para python3.8
# FIX: esta build de python3.8 (deadsnakes) no trae pip ni ensurepip
# incluidos. python3.8-venv trae ambos.
!apt-get update
!apt-get install -y python3.8-dev python3.8-distutils python3.8-venv
!python3.8 -m ensurepip --upgrade

In [ ]:
# Celda 5: confirmar pip de python3.8 (debe decir 'python 3.8')
!python3.8 -m pip --version

In [ ]:
# Celda 6: instalar dependencias de SadTalker
!python3.8 -m pip install "setuptools==59.5.0" wheel
!python3.8 -m pip install -r requirements.txt --no-build-isolation

In [ ]:
# Celda 7: FIX — parche de compatibilidad basicsr <-> torchvision nuevo
# basicsr fue escrito para una versión vieja de torchvision que tenía
# el módulo functional_tensor (eliminado en versiones nuevas).
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.8/dist-packages/basicsr/data/degradations.py

In [ ]:
# Celda 8: descargar checkpoints de SadTalker (script oficial)
!bash scripts/download_models.sh

In [ ]:
# Celda 9: FIX — epoch_20.pth NO está en el release v0.0.2-rc que usa
# download_models.sh. Hay que bajarlo del release v0.0.2 (sin '-rc').
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/epoch_20.pth -P ./checkpoints
!ls -la checkpoints/

In [ ]:
# Celda 10: volver a la raíz del proyecto para poder importar src/
%cd ..
!pwd

In [ ]:
# Celda 11: instalar librerías del notebook principal
# FIX: usar 'python -m pip', NO 'pip' a secas — el ensurepip de la
# celda 4 deja el comando 'pip' apuntando a python3.8 por error, y
# 'pip' a secas instalaría todo en el intérprete equivocado.
!python -m pip install openai gTTS diffusers transformers accelerate

In [ ]:
# Celda 12: importar los módulos del proyecto
from src.orquestador import init_lola
from src.config import SCRIPT_DIA_1, RUTA_IMAGEN_ANCLA

In [ ]:
# Celda 13: generar el video
# FIX: inference.py de SadTalker debe correrse parado en la carpeta
# SadTalker/, por eso el os.chdir. La imagen usa ruta ABSOLUTA para
# que funcione sin importar la carpeta actual.
# guion_id identifica el video final (./results/dia_1.mp4) y hace que
# re-correr esta celda no regenere nada si ese archivo ya existe.
import os
os.chdir('/content/lola-project/SadTalker')

res = init_lola(
    SCRIPT_DIA_1,
    guion_id="dia_1",
    reusar_imagen='/content/lola-project/examples/source_image/lola_512.png'
)
print(res)

In [ ]:
# Celda 14: ver el resultado
from IPython.display import Video
Video(res, embed=True, width=400)

In [ ]:
# Celda 15: descargar el video
from google.colab import files
files.download(res)

In [ ]:
# Celda 15b: generar el video del día 3
# Mismo patrón que la Celda 13, con el guion de SCRIPT_DIA_3. No hace
# falta repetir el clone/setup — solo el os.chdir, por si esta celda
# se corre sin haber corrido la 13 antes en esta sesión.
# Después de correr esto, volvé a correr las Celdas 14 y 15 para ver
# y descargar ESTE video (usan 'res', que ahora apunta acá).
import os
from src.config import SCRIPT_DIA_3
os.chdir('/content/lola-project/SadTalker')

res = init_lola(
    SCRIPT_DIA_3,
    guion_id="dia_3",
    reusar_imagen='/content/lola-project/examples/source_image/lola_512.png'
)
print(res)

## Si algo falla

Usá la función de diagnóstico para ver el error completo en vez de que
se pierda en el output largo de la celda:

```python
from src.animacion import diagnosticar_error
resultado = diagnosticar_error(
    "/content/lola-project/examples/source_image/lola_512.png",
    "examples/driven_audio/lola_audio.wav"
)
```

Y si aparece un `ModuleNotFoundError` de un archivo específico de
SadTalker (checkpoint faltante), buscá en qué release de GitHub está:

```python
!curl -sL https://api.github.com/repos/OpenTalker/SadTalker/releases | grep -E "tag_name|\"name\""
```